# 1. Language Detection

In [ ]:
# Paths are relative to the notebooks/ folder of this repository.
# On Google Colab, clone the repository first and run from notebooks/:
#   !git clone https://github.com/Syukrondzeko/natural-disaster-app-review-analysis.git
#   %cd natural-disaster-app-review-analysis/notebooks
DATA_DIR = "../data"

In [ ]:
import pandas as pd

path = f"{DATA_DIR}/1_raw/final_raw_df.xlsx"

df = pd.read_excel(path)

df.head()


,id,app,content,score,source
0,1,disaster alert,Good App,5,play_store
1,2,disaster alert,"Working as a Public Health Nurse, I get to res...",5,play_store
2,3,disaster alert,"Nice to have, before traveling to unknown terr...",5,play_store
3,4,disaster alert,I like! I'm trying to find anything about tsun...,5,play_store
4,5,disaster alert,Not accurate,1,play_store


In [ ]:
import re

print(f">> Original rows: {len(df)}")

# --------------------------------------------------
# Emoji removal function
# --------------------------------------------------
def remove_emoji(text):
    if not isinstance(text, str):
        return ""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\u2600-\u26FF"          # misc symbols
        "\u2700-\u27BF"          # dingbats
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub("", text)

# --------------------------------------------------
# Step 1: Remove emojis
# --------------------------------------------------
df["content"] = df["content"].apply(remove_emoji)

# --------------------------------------------------
# Step 2: Remove empty / whitespace-only reviews
# --------------------------------------------------
rows_before = len(df)

df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"] != ""]

rows_after = len(df)
deleted_rows = rows_before - rows_after

# --------------------------------------------------
# Report
# --------------------------------------------------
print(f">> Rows after cleaning: {rows_after}")
print(f">> Rows deleted (empty after emoji removal): {deleted_rows}")


>> Original rows: 60473
>> Rows after cleaning: 60244
>> Rows deleted (empty after emoji removal): 229


In [ ]:
pip install -q langid

In [ ]:
pip install -q pycountry

In [ ]:
import langid
import pycountry

def detect_language(text):
    try:
        lang, score = langid.classify(text)
        language_name = pycountry.languages.get(alpha_2=lang)
        return language_name.name if language_name else lang
    except:
        return "unknown"


df["language"] = df["content"].astype(str).apply(detect_language)

In [ ]:
language_counts = (
    df.groupby("language")
      .size()
      .reset_index(name="count")
      .sort_values(by="count", ascending=False)
)

for _, row in language_counts.iterrows():
    print(f"{row['language']}: {row['count']}")



English: 52297
Japanese: 2619
German: 1033
Spanish: 1031
Turkish: 907
Dutch: 294
French: 260
Italian: 225
Swedish: 213
Chinese: 122
Danish: 107
Finnish: 80
Portuguese: 79
Persian: 72
Slovak: 56
Arabic: 49
Tagalog: 47
Polish: 45
Indonesian: 43
Maltese: 39
Bengali: 36
Estonian: 36
Norwegian Bokmål: 35
Galician: 34
Norwegian: 28
Georgian: 26
Catalan: 24
Azerbaijani: 24
Slovenian: 23
Malagasy: 23
Romanian: 22
Malay (macrolanguage): 20
Basque: 20
Mongolian: 17
Lithuanian: 15
Czech: 13
Welsh: 10
Norwegian Nynorsk: 10
Russian: 9
Hungarian: 9
Swahili (macrolanguage): 9
Urdu: 9
Korean: 9
Oriya (macrolanguage): 8
Amharic: 8
Thai: 8
Hindi: 8
Croatian: 8
Latin: 7
Bosnian: 7
Khmer: 6
Bulgarian: 6
Afrikaans: 6
Albanian: 6
Nepali (macrolanguage): 6
Kinyarwanda: 5
Breton: 5
Esperanto: 5
Armenian: 4
Haitian: 4
Walloon: 4
Latvian: 4
Tamil: 4
Faroese: 3
Marathi: 3
Irish: 3
Modern Greek (1453-): 3
Occitan (post 1500): 3
Volapük: 3
Xhosa: 2
Belarusian: 2
Vietnamese: 2
Uighur: 2
Kannada: 2
Gujarati: 2
Dzong

# 2. Language Translation

## 2.1. Japan to English

In [ ]:
print(">> Step 1: Creating Japanese-only dataframe...")
japan_df = df[df["language"] == "Japanese"].copy()
print(f">> Japanese rows: {len(japan_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
model_name = "Helsinki-NLP/opus-mt-ja-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM allows
texts = japan_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(range(0, len(texts), batch_size), desc="Translating Japanese reviews"):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

japan_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")





In [ ]:
import pandas as pd

japan_df.to_excel(
    f"{DATA_DIR}/2_translated/japanese_reviews_translated.xlsx",
    index=False
)

## 2.2. Chinesse to English

In [ ]:
print(">> Step 1: Creating Chinese-only dataframe...")
chinese_df = df[df["language"] == "Chinese"].copy()
print(f">> Chinese rows: {len(chinese_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Chinese → English model
model_name = "Helsinki-NLP/opus-mt-zh-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM allows
texts = chinese_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Chinese reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

chinese_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

chinese_df.to_excel(
    f"{DATA_DIR}/2_translated/chinesse_reviews_translated.xlsx",
    index=False
)



>> Step 1: Creating Chinese-only dataframe...
>> Chinese rows: 122
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/805k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/807k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Chinese reviews:  25%|██▌       | 2/8 [00:00<00:01,  4.99it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Translating Chinese reviews: 100%|██████████| 8/8 [00:09<00:00,  1.20s/it]

>> Step 6: Translation finished.


## 2.3. Korean to English

In [ ]:
print(">> Step 1: Creating Korean-only dataframe...")
korean_df = df[df["language"] == "Korean"].copy()
print(f">> Korean rows: {len(korean_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Korean → English model
model_name = "Helsinki-NLP/opus-mt-ko-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = korean_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Korean reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

korean_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

korean_df.to_excel(
    f"{DATA_DIR}/2_translated/korean_reviews_translated.xlsx",
    index=False
)



>> Step 1: Creating Korean-only dataframe...
>> Korean rows: 9
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...



Translating Korean reviews: 100%|██████████| 1/1 [00:04<00:00,  4.39s/it]

>> Step 6: Translation finished.


## 2.4. Turkey to English

In [ ]:
print(">> Step 1: Creating Turkish-only dataframe...")
turkish_df = df[df["language"] == "Turkish"].copy()
print(f">> Turkish rows: {len(turkish_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Turkish → English model
model_name = "Helsinki-NLP/opus-mt-tr-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = turkish_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Turkish reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

turkish_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

turkish_df.to_excel(
    f"{DATA_DIR}/2_translated/turkish_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Turkish-only dataframe...
>> Turkish rows: 907
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/840k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Turkish reviews:   0%|          | 0/57 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

Translating Turkish reviews: 100%|██████████| 57/57 [03:10<00:00,  3.33s/it]

>> Step 6: Translation finished.


## 2.5. Spanish to English

In [ ]:
print(">> Step 1: Creating Spanish-only dataframe...")
spanish_df = df[df["language"] == "Spanish"].copy()
print(f">> Spanish rows: {len(spanish_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Spanish → English model
model_name = "Helsinki-NLP/opus-mt-es-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = spanish_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Spanish reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

spanish_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

spanish_df.to_excel(
    f"{DATA_DIR}/2_translated/spanish_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Spanish-only dataframe...
>> Spanish rows: 1031
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Spanish reviews:   0%|          | 0/65 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Translating Spanish reviews: 100%|██████████| 65/65 [00:53<00:00,  1.21it/s]


>> Step 6: Translation finished.


## 2.6. German to English

In [ ]:
print(">> Step 1: Creating German-only dataframe...")
german_df = df[df["language"] == "German"].copy()
print(f">> German rows: {len(german_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# German → English model
model_name = "Helsinki-NLP/opus-mt-de-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = german_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating German reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

german_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

german_df.to_excel(
    f"{DATA_DIR}/2_translated/german_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating German-only dataframe...
>> German rows: 1033
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating German reviews:   0%|          | 0/65 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

Translating German reviews: 100%|██████████| 65/65 [01:12<00:00,  1.12s/it]


>> Step 6: Translation finished.


## 2.7. Dutch to English

In [ ]:
print(">> Step 1: Creating Dutch-only dataframe...")
dutch_df = df[df["language"] == "Dutch"].copy()
print(f">> Dutch rows: {len(dutch_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Dutch → English model
model_name = "Helsinki-NLP/opus-mt-nl-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = dutch_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Dutch reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

dutch_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

dutch_df.to_excel(
    f"{DATA_DIR}/2_translated/dutch_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Dutch-only dataframe...
>> Dutch rows: 294
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Dutch reviews:   0%|          | 0/19 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/316M [00:00<?, ?B/s]

Translating Dutch reviews: 100%|██████████| 19/19 [00:20<00:00,  1.08s/it]

>> Step 6: Translation finished.


## 2.8. French to English

In [ ]:
print(">> Step 1: Creating French-only dataframe...")
french_df = df[df["language"] == "French"].copy()
print(f">> French rows: {len(french_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# French → English model
model_name = "Helsinki-NLP/opus-mt-fr-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = french_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating French reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

french_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

french_df.to_excel(
    f"{DATA_DIR}/2_translated/french_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating French-only dataframe...
>> French rows: 260
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...


Translating French reviews: 100%|██████████| 17/17 [00:17<00:00,  1.01s/it]


>> Step 6: Translation finished.


## 2.9. Italian to English

In [ ]:
print(">> Step 1: Creating Italian-only dataframe...")
italian_df = df[df["language"] == "Italian"].copy()
print(f">> Italian rows: {len(italian_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Italian → English model
model_name = "Helsinki-NLP/opus-mt-it-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = italian_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Italian reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

italian_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

italian_df.to_excel(
    f"{DATA_DIR}/2_translated/italian_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Italian-only dataframe...
>> Italian rows: 225
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...



Translating Italian reviews: 100%|██████████| 15/15 [00:33<00:00,  2.22s/it]

>> Step 6: Translation finished.


## 2.10. Swedish to English

In [ ]:
print(">> Step 1: Creating Swedish-only dataframe...")
swedish_df = df[df["language"] == "Swedish"].copy()
print(f">> Swedish rows: {len(swedish_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Swedish → English model
model_name = "Helsinki-NLP/opus-mt-sv-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = swedish_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Swedish reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

swedish_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

swedish_df.to_excel(
    f"{DATA_DIR}/2_translated/swedish_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Swedish-only dataframe...
>> Swedish rows: 213
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/815k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Swedish reviews:  36%|███▌      | 5/14 [00:00<00:01,  6.51it/s]

model.safetensors:   0%|          | 0.00/294M [00:00<?, ?B/s]

Translating Swedish reviews: 100%|██████████| 14/14 [00:02<00:00,  4.67it/s]


>> Step 6: Translation finished.


## 2.11. Danish to English

In [ ]:
print(">> Step 1: Creating Danish-only dataframe...")
danish_df = df[df["language"] == "Danish"].copy()
print(f">> Danish rows: {len(danish_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Danish → English model
model_name = "Helsinki-NLP/opus-mt-da-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # increase to 32 if RAM/GPU allows
texts = danish_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Danish reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

danish_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

danish_df.to_excel(
    f"{DATA_DIR}/2_translated/danish_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Danish-only dataframe...
>> Danish rows: 107
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/820k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/788k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Danish reviews:  29%|██▊       | 2/7 [00:00<00:01,  4.62it/s]

model.safetensors:   0%|          | 0.00/300M [00:00<?, ?B/s]

Translating Danish reviews: 100%|██████████| 7/7 [00:01<00:00,  5.21it/s]

>> Step 6: Translation finished.


## 2.12. Croatian to English

In [ ]:
print(">> Step 1: Creating Croatian-only dataframe...")
croatian_df = df[df["language"] == "Croatian"].copy()
print(f">> Croatian rows: {len(croatian_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Croatian → English (via Slavic model)
model_name = "Helsinki-NLP/opus-mt-sla-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = croatian_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Croatian reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

croatian_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

croatian_df.to_excel(
    f"{DATA_DIR}/2_translated/croatian_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Croatian-only dataframe...
>> Croatian rows: 8
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/860k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...


Translating Croatian reviews: 100%|██████████| 1/1 [00:00<00:00,  3.30it/s]

>> Step 6: Translation finished.


## 2.13. Finnish to English

In [ ]:
print(">> Step 1: Creating Finnish-only dataframe...")
finnish_df = df[df["language"] == "Finnish"].copy()
print(f">> Finnish rows: {len(finnish_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Finnish → English model
model_name = "Helsinki-NLP/opus-mt-fi-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = finnish_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Finnish reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

finnish_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

finnish_df.to_excel(
    f"{DATA_DIR}/2_translated/finnish_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Finnish-only dataframe...
>> Finnish rows: 80
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/832k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Translating Finnish reviews:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Translating Finnish reviews: 100%|██████████| 5/5 [00:00<00:00,  5.74it/s]

>> Step 6: Translation finished.


## 2.14. Persian to English

In [ ]:
print(">> Step 1: Creating Persian-only dataframe...")
persian_df = df[df["language"] == "Persian"].copy()
print(f">> Persian rows: {len(persian_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Persian → English (via multilingual model)
model_name = "Helsinki-NLP/opus-mt-mul-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = persian_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Persian reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

persian_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

persian_df.to_excel(
    f"{DATA_DIR}/2_translated/persian_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Persian-only dataframe...
>> Persian rows: 72
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Persian reviews:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/310M [00:00<?, ?B/s]

Translating Persian reviews: 100%|██████████| 5/5 [00:03<00:00,  1.31it/s]


>> Step 6: Translation finished.


## 2.15. Portuguese to English

In [ ]:
print(">> Step 1: Creating Portuguese-only dataframe...")
portuguese_df = df[df["language"] == "Portuguese"].copy()
print(f">> Portuguese rows: {len(portuguese_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Portuguese → English (via Romance languages model)
model_name = "Helsinki-NLP/opus-mt-roa-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = portuguese_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Portuguese reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

portuguese_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

portuguese_df.to_excel(
    f"{DATA_DIR}/2_translated/portuguese_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Portuguese-only dataframe...
>> Portuguese rows: 79
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...


Translating Portuguese reviews: 100%|██████████| 5/5 [00:04<00:00,  1.12it/s]

>> Step 6: Translation finished.


## 2.16. Arabic to English

In [ ]:
print(">> Step 1: Creating Arabic-only dataframe...")
arabic_df = df[df["language"] == "Arabic"].copy()
print(f">> Arabic rows: {len(arabic_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Arabic → English model
model_name = "Helsinki-NLP/opus-mt-ar-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = arabic_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Arabic reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

arabic_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

arabic_df.to_excel(
    f"{DATA_DIR}/2_translated/arabic_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Arabic-only dataframe...
>> Arabic rows: 49
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Arabic reviews:   0%|          | 0/4 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Translating Arabic reviews: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]

>> Step 6: Translation finished.


## 2.17. Tagalog to English

In [ ]:
print(">> Step 1: Creating Tagalog-only dataframe...")
tagalog_df = df[df["language"] == "Tagalog"].copy()
print(f">> Tagalog rows: {len(tagalog_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Tagalog → English model
model_name = "Helsinki-NLP/opus-mt-tl-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = tagalog_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Tagalog reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

tagalog_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

tagalog_df.to_excel(
    f"{DATA_DIR}/2_translated/tagalog_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Tagalog-only dataframe...
>> Tagalog rows: 47
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/834k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (4 outputs left out)

Translating Tagalog reviews:  33%|███▎      | 1/3 [00:00<00:00,  2.56it/s]

model.safetensors:   0%|          | 0.00/296M [00:00<?, ?B/s]

Translating Tagalog reviews: 100%|██████████| 3/3 [00:04<00:00,  1.65s/it]

>> Step 6: Translation finished.


## 2.18. Bengali to English

In [ ]:
print(">> Step 1: Creating Bengali-only dataframe...")
bengali_df = df[df["language"] == "Bengali"].copy()
print(f">> Bengali rows: {len(bengali_df)}")

print(">> Step 2: Importing libraries...")
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm

print(">> Step 3: Loading translation model...")
# Bengali → English model
model_name = "Helsinki-NLP/opus-mt-bn-en"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.eval()  # IMPORTANT

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f">> Model loaded on device: {device}")

# --------------------------------------------------
# Batch translation function
# --------------------------------------------------
print(">> Step 4: Defining batch translation function...")

def translate_batch(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs)

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# --------------------------------------------------
# Apply translation in batches
# --------------------------------------------------
print(">> Step 5: Starting batch translation...")

batch_size = 16  # reduce to 8 if GPU memory is tight
texts = bengali_df["content"].astype(str).tolist()
translated_texts = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Translating Bengali reviews"
):
    batch = texts[i:i + batch_size]
    translated_texts.extend(translate_batch(batch))

bengali_df["content_translated"] = translated_texts

print(">> Step 6: Translation finished.")

import pandas as pd

bengali_df.to_excel(
    f"{DATA_DIR}/2_translated/bengali_reviews_translated.xlsx",
    index=False
)


>> Step 1: Creating Bengali-only dataframe...
>> Bengali rows: 36
>> Step 2: Importing libraries...
>> Step 3: Loading translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/806k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

>> Model loaded on device: cuda
>> Step 4: Defining batch translation function...
>> Step 5: Starting batch translation...


Translating Bengali reviews: 100%|██████████| 3/3 [00:00<00:00,  4.07it/s]

>> Step 6: Translation finished.


# Counting Translated Language

In [ ]:
translation_list = ["Japanese", "German", "Spanish", "Turkish", "Dutch", "French",
                    "Italian", "Swedish", "Chinese", "Danish", "Finnish", "Portuguese",
                    "Persian", "Arabic", "Tagalog", "Bengali", "Croatian", "Korean"
                    ]
translated_df = df[df["language"].isin(translation_list)]
total_rows = len(df)
translated_rows = len(translated_df)
not_translated_rows = total_rows - translated_rows

print(f"Total rows in df: {total_rows}")
print(f"Translated rows: {translated_rows}")
print(f"Not translated rows: {not_translated_rows}")


Total rows in df: 60244
Translated rows: 7191
Not translated rows: 53053


# Concatenation

In [ ]:
import pandas as pd
import glob

translated_path = f"{DATA_DIR}/2_translated/*_reviews_translated.xlsx"
files = glob.glob(translated_path)

print(f">> Found {len(files)} translated files")

translated_dfs = []

for file in files:
    print(f">> Loading: {file}")
    translated_dfs.append(pd.read_excel(file))

translated_df = pd.concat(translated_dfs, ignore_index=True)

print(f">> Total translated rows: {len(translated_df)}")



>> Found 18 translated files
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/japanese_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/chinesse_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/korean_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/turkish_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/spanish_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/german_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/dutch_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/french_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/italian_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Dataset/swedish_reviews_translated.xlsx
>> Loading: /content/drive/MyDrive/Natural Disaster/Da

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60244 entries, 0 to 60472
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        60244 non-null  int64 
 1   app       60244 non-null  object
 2   content   60244 non-null  object
 3   score     60244 non-null  int64 
 4   source    60244 non-null  object
 5   language  60244 non-null  object
dtypes: int64(2), object(4)
memory usage: 3.2+ MB


In [ ]:
import pandas as pd

# ---------------------------------------
# Step 0: Languages that were translated
# ---------------------------------------
translation_list = [
    "Japanese", "German", "Spanish", "Turkish", "Dutch", "French",
    "Italian", "Swedish", "Chinese", "Danish", "Finnish", "Portuguese",
    "Persian", "Arabic", "Tagalog", "Bengali", "Croatian", "Korean"
]

# ---------------------------------------
# Step 1: Copy original df
# ---------------------------------------
copy_df = df.copy()
original_len = len(df)

# ---------------------------------------
# Step 2: Drop rows that were translated
# ---------------------------------------
copy_df = copy_df[~copy_df["language"].isin(translation_list)]

# ---------------------------------------
# Step 3: Create translated column (fallback)
# ---------------------------------------
copy_df["content_translated"] = copy_df["content"]

# ---------------------------------------
# Step 4: Concatenate with translated_df
# ---------------------------------------
final_df = pd.concat(
    [copy_df, translated_df],
    ignore_index=True
)

# ---------------------------------------
# Step 5: Reset index
# ---------------------------------------
final_df = final_df.reset_index(drop=True)

# ---------------------------------------
# Step 6: Sanity checks
# ---------------------------------------
print(f">> Original df length: {original_len}")
print(f">> Final df length: {len(final_df)}")
print(f">> Lengths match: {original_len == len(final_df)}")

print("\n>> Translation usage by language:")
print(final_df["language"].value_counts())

print("\n>> Missing translated_content:", final_df["content_translated"].isna().sum())


>> Original df length: 60244
>> Final df length: 60244
>> Lengths match: True

>> Translation usage by language:
language
English      52297
Japanese      2619
German        1033
Spanish       1031
Turkish        907
             ...  
Pushto           1
Javanese         1
Malayalam        1
Icelandic        1
Lao              1
Name: count, Length: 91, dtype: int64

>> Missing translated_content: 11


In [ ]:
missing_df = final_df[final_df["content_translated"].isna()]

print(f">> Missing translated_content rows: {len(missing_df)}")
missing_df


>> Missing translated_content rows: 11


,id,app,content,score,source,language,content_translated
58781,1369,earthquake alert!,Doesn't work,1,play_store,Dutch,NaN
58835,8683,earthquake alert!,Doesn't work. Uninstalled. EVO,1,play_store,Dutch,NaN
58843,10088,floods near me nsw,Doesn't work,1,play_store,Dutch,NaN
58895,18954,natural disaster monitor,Doesn't work,1,play_store,Dutch,NaN
58905,19855,sd emergency,Doesn't work,1,play_store,Dutch,NaN
58907,19952,hazards near me nsw,Doesn't work,1,play_store,Dutch,NaN
58922,22516,vicemergency,Doesn't work,1,play_store,Dutch,NaN
58951,29169,eqinfo,Doesn't work,1,play_store,Dutch,NaN
58972,33023,lastquake,Doesn't work,2,play_store,Dutch,NaN
58981,33402,my earthquake alerts,Doesn't work,1,play_store,Dutch,NaN


In [ ]:
final_df["content_translated"] = final_df["content_translated"].fillna(final_df["content"])
final_df["content_translated"].isna().sum()


np.int64(0)

In [ ]:
# output_path = f"{DATA_DIR}/2_translated/final_translated_df.xlsx"

# final_df.to_excel(output_path, index=False)

# print(f">> Saved final translated dataset to: {output_path}")


>> Saved final translated dataset to: /content/drive/MyDrive/Natural Disaster/Dataset/final_translated_df.xlsx
